# 04 - Feature Engineering

Transform the cleaned transaction, loan, and customer data into
a single feature matrix — one row per customer — for use in the
segmentation model.

Each feature is designed to capture a specific dimension of customer
behaviour relevant to the post-pivot segmentation question.

In [1]:
# Set up
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
from settings import PATHS, apply_pandas_settings

apply_pandas_settings()

In [2]:
customers = pd.read_csv(PATHS['clean_customers'], parse_dates = ['account_open_date'])
loans = pd.read_csv(PATHS['clean_loans'])
transactions = pd.read_csv(PATHS['clean_transactions'], parse_dates = ['transaction_date'])

In [3]:
# Define crucial dates as constants, following software engineering DRY principle
PIVOT_DATE = pd.Timestamp('2022-07-01')
END_DATE = transactions['transaction_date'].max() + pd.Timedelta(days = 1)

## Feature 1 - RFM (Recency, Frequency, Monetary)

Calculated for the post-pivot period only (Jul–Dec 2022).
Recency is measured in days from the customer's last transaction
to the end of the dataset (Dec 31 2022) — higher recency means
less recent activity.

In [4]:
# Build RFM features post-pivot
post_pivot = transactions[transactions['transaction_date'] >= PIVOT_DATE]

rfm_features = post_pivot.groupby('customer_id').agg(
    last_txn_date = ('transaction_date', 'max'),
    frequency = ('transaction_id', 'count'),
    monetary = ('amount', 'sum')
)

rfm_features['recency'] = (END_DATE - rfm_features['last_txn_date']).dt.days

rfm_features = rfm_features.drop(columns = ['last_txn_date']).reset_index()

rfm_features.shape

(3410, 4)

In [5]:
# Include churned customers with no post-pivot activity
customer_base = customers[['customer_id']].copy()

rfm_features = customer_base.merge(rfm_features, on = 'customer_id', how = 'left')

max_recency = int((END_DATE - PIVOT_DATE).days)

rfm_features['frequency'] = rfm_features['frequency'].fillna(0).astype(int)
rfm_features['monetary'] = rfm_features['monetary'].fillna(0)
rfm_features['recency'] = rfm_features['recency'].fillna(max_recency).astype(int)

rfm_features.shape

(5000, 4)

In [6]:
# Verify rfm_features
print('Summary statistics')
display(rfm_features[['frequency',
                      'monetary',
                      'recency']].describe().round(2))
print(f"\nRFM feature matrix shape: {rfm_features.shape}")
print(f"Max recency (churned customers): {max_recency} days")
print(f"\nNull values remaining:")
print(rfm_features.isnull().sum())

Summary statistics


,frequency,monetary,recency
count,5000.00,5000.00,5000.00
mean,3.55,136129.33,92.48
std,4.53,202603.22,74.01
min,0.00,0.00,1.00
25%,0.00,0.00,19.00
50%,1.00,36650.00,74.00
75%,6.00,206250.00,184.00
max,18.00,1251400.00,184.00



RFM feature matrix shape: (5000, 4)
Max recency (churned customers): 184 days

Null values remaining:
customer_id    0
frequency      0
monetary       0
recency        0
dtype: int64


In [7]:
rfm_features.head(10)

,customer_id,frequency,monetary,recency
0,CUS00001,11,351800.00,6
1,CUS00002,1,90250.00,27
2,CUS00003,0,0.00,184
3,CUS00004,15,1122650.00,13
4,CUS00005,0,0.00,184
5,CUS00006,1,90200.00,161
6,CUS00007,0,0.00,184
7,CUS00008,8,224250.00,29
8,CUS00009,3,19650.00,42
9,CUS00010,0,0.00,184


## Feature 1 - RFM Summary

| Feature | Description | Min | Median | Max |
|---|---|---|---|---|
| recency | Days from last post-pivot txn to Dec 31 | 1 | 74 | 184 |
| frequency | Post-pivot transaction count | 0 | 1 | 18 |
| monetary | Total post-pivot spend in ₦ | 0 | 36,650 | 1,251,400 |

1,590 churned customers included with frequency=0, monetary=0,
and recency=184 days (maximum — full post-pivot period of inactivity).

## Feature 2 - Pre-Pivot Frequency and Pivot Ratio

Pre-pivot frequency captures engagement before the pivot.
Pivot ratio compares post-pivot to pre-pivot activity -
a ratio of 0 indicates complete churn, above 1 indicates
stronger post-pivot engagement than pre-pivot.

In [8]:
# Pre pivot total_transaction_count and pivot ratio

# pre_pivot transactions
pre_pivot = transactions[transactions['transaction_date'] < PIVOT_DATE]

# pre_pivot transaction frequency per customer
pre_freq = pre_pivot.groupby('customer_id').size().reset_index(name = 'pre_pivot_frequency')

# ensure all customers are included
pre_freq = customer_base.merge(pre_freq, on = 'customer_id', how = 'left')

# Handle customers with no pre_pivot activity
pre_freq['pre_pivot_frequency'] = pre_freq['pre_pivot_frequency'].fillna(0).astype(int)

# Retrieve the frequency feature to compute the pivot ratio
pre_freq = pre_freq.merge(rfm_features[['customer_id', 'frequency']], on = 'customer_id', how = 'left')

# compute the pivot ratio feature
pre_freq['pivot_ratio'] = (
    pre_freq['frequency'] / pre_freq['pre_pivot_frequency']
)

# handle pivot_ratio values where pre_pivot_frequency = 0
pre_freq['pivot_ratio'] = (
    pre_freq['frequency'] /
    (pre_freq['pre_pivot_frequency'] + 1)
).round(4)

# remove the frequency feature as it already exist in rfm_features
pre_freq = pre_freq.drop(columns = ['frequency'])

In [9]:
# Verify pre_freq features
print(f"Shape: {pre_freq.shape}")
print(f"\nNull values:")
print(pre_freq.isnull().sum())
print(f"\nSummary:")
display(pre_freq[['pre_pivot_frequency', 'pivot_ratio']].describe().round(2))

Shape: (5000, 3)

Null values:
customer_id            0
pre_pivot_frequency    0
pivot_ratio            0
dtype: int64

Summary:


,pre_pivot_frequency,pivot_ratio
count,5000.00,5000.00
mean,6.29,0.46
std,3.62,0.58
min,1.00,0.00
25%,3.00,0.00
50%,6.00,0.25
75%,9.00,0.67
max,15.00,3.60


In [10]:
pre_freq.head(10)

,customer_id,pre_pivot_frequency,pivot_ratio
0,CUS00001,10,1.00
1,CUS00002,10,0.09
2,CUS00003,6,0.00
3,CUS00004,9,1.50
4,CUS00005,3,0.00
5,CUS00006,4,0.20
6,CUS00007,1,0.00
7,CUS00008,10,0.73
8,CUS00009,2,1.00
9,CUS00010,3,0.00


## Feature 2 - Pre-Pivot Frequency and Pivot Ratio Summary

| Feature | Description | Min | Median | Max |
|---|---|---|---|---|
| pre_pivot_frequency | Number of pre-pivot transactions per customer | 1 | 6 | 15 |
| pivot_ratio | Post-pivot frequency / (pre-pivot frequency + 1) | 0.00 | 0.25 | 3.60 |

**Notes:**
- Every customer had at least 1 pre-pivot transaction
- Median pivot ratio of 0.25 means the typical customer's post-pivot
  activity fell to 25% of their pre-pivot level
- At least 25% of customers have a pivot ratio of 0 — complete churn
- A small group with ratio > 1 actually became more active post-pivot
  — these are the customers who genuinely adopted the transactional product

## Feature 3 - Product Diversity

Number of distinct transaction types used per customer
in the post-pivot period. Higher diversity indicates
broader platform engagement and stronger retention signal.

In [11]:
# Product diversity

# Count unique transaction types per customer
product_diversity = (post_pivot.groupby('customer_id')['transaction_type']
                     .nunique()
                     .reset_index(name = 'product_diversity'))

# Include all customers
product_diversity = customer_base.merge(product_diversity, on = 'customer_id', how = 'left')

# Handle customers with no product diversity
product_diversity['product_diversity'] = product_diversity['product_diversity'].fillna(0).astype(int)


# Verify product_diversity feature
print(f"Shape: {product_diversity.shape}")
print(f"\nNull values:")
print(product_diversity.isnull().sum())
print(f"\nProduct diversity distribution:")
display(product_diversity["product_diversity"].value_counts().sort_index())

Shape: (5000, 2)

Null values:
customer_id          0
product_diversity    0
dtype: int64

Product diversity distribution:


product_diversity
0    1590
1    1246
2     521
3     661
4     717
5     161
6     104
Name: count, dtype: int64

In [12]:
product_diversity.head()

,customer_id,product_diversity
0,CUS00001,4
1,CUS00002,1
2,CUS00003,0
3,CUS00004,5
4,CUS00005,0


## Feature 3 - Product Diversity Summary

| Score | Customers | Interpretation |
|---|---|---|
| 0 | 1,590 | Churned — no post-pivot activity |
| 1 | 1,246 | Single product — fragile retention |
| 2 | 521 | Two products — limited engagement |
| 3 | 661 | Three products — moderate engagement |
| 4 | 717 | Four products — strong engagement |
| 5 | 161 | Five products — high engagement |
| 6 | 104 | Six products — maximum engagement, Tier 3 only |


## Feature 4 — Loan Behaviour

Two binary features derived from the loans table:
- loan_taken: 1 if the customer took a loan, 0 otherwise
- loan_defaulted: 1 if the customer defaulted, 0 otherwise
  (customers who never took a loan are also assigned 0)

In [13]:
# Loan behaviour

loan_features = loans[['customer_id', 'loan_taken', 'loan_defaulted']].copy()

loan_features['loan_taken'] = (loan_features['loan_taken'] == 'Yes').astype(int)
loan_features['loan_defaulted'] = (loan_features['loan_defaulted'] == 'Yes').astype(int)



In [14]:
print(f"Shape: {loan_features.shape}")
print(f"\nNull values:")
print(loan_features.isnull().sum())
print(f"\nloan_taken distribution:")
display(loan_features["loan_taken"].value_counts())
print(f"\nloan_defaulted distribution:")
display(loan_features["loan_defaulted"].value_counts())

Shape: (5000, 3)

Null values:
customer_id       0
loan_taken        0
loan_defaulted    0
dtype: int64

loan_taken distribution:


loan_taken
1    2795
0    2205
Name: count, dtype: int64


loan_defaulted distribution:


loan_defaulted
0    3602
1    1398
Name: count, dtype: int64

## Feature 4 — Loan Behaviour Summary

| Feature | Value | Count | Interpretation |
|---|---|---|---|
| loan_taken | 1 | 2,795 | Took a loan |
| loan_taken | 0 | 2,205 | Never took a loan |
| loan_defaulted | 1 | 1,398 | Defaulted on loan |
| loan_defaulted | 0 | 3,602 | Repaid loan or never borrowed |

Both features encoded as binary (0/1).
NaN values in loan_defaulted correctly resolved to 0
via pandas comparison behaviour — no separate handling required.

## Feature Matrix

Combine all engineered features into a single matrix
of 5,000 rows (one per customer) for use in the
segmentation model.

In [15]:
# Create feature matrix per customer

feature_matrix = (customer_base.merge(rfm_features, on = 'customer_id', how = 'left')
                  .merge(pre_freq, on = 'customer_id', how = 'left')
                  .merge(product_diversity, on = 'customer_id', how = 'left')
                  .merge(loan_features, on = 'customer_id', how = 'left'))

In [16]:
feature_matrix.head()

,customer_id,frequency,monetary,recency,pre_pivot_frequency,pivot_ratio,product_diversity,loan_taken,loan_defaulted
0,CUS00001,11,351800.00,6,10,1.00,4,0,0
1,CUS00002,1,90250.00,27,10,0.09,1,1,0
2,CUS00003,0,0.00,184,6,0.00,0,1,0
3,CUS00004,15,1122650.00,13,9,1.50,5,0,0
4,CUS00005,0,0.00,184,3,0.00,0,1,1


In [17]:
# ── Final verification ──
print(f"Feature matrix shape     : {feature_matrix.shape}")
print(f"Expected shape           : (5000, 9)")
print(f"Shape correct            : {feature_matrix.shape == (5000, 9)}")
print(f"\nNull values              : {feature_matrix.isnull().sum().sum()}")
print(f"\nFeature summary:")
display(feature_matrix.describe().round(2))

Feature matrix shape     : (5000, 9)
Expected shape           : (5000, 9)
Shape correct            : True

Null values              : 0

Feature summary:


,frequency,monetary,recency,pre_pivot_frequency,pivot_ratio,product_diversity,loan_taken,loan_defaulted
count,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00
mean,3.55,136129.33,92.48,6.29,0.46,1.71,0.56,0.28
std,4.53,202603.22,74.01,3.62,0.58,1.66,0.50,0.45
min,0.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00
25%,0.00,0.00,19.00,3.00,0.00,0.00,0.00,0.00
50%,1.00,36650.00,74.00,6.00,0.25,1.00,1.00,0.00
75%,6.00,206250.00,184.00,9.00,0.67,3.00,1.00,1.00
max,18.00,1251400.00,184.00,15.00,3.60,6.00,1.00,1.00


In [18]:
# Save engineered features

feature_matrix.to_csv(PATHS['features'], index = False)